In [1]:
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms.v2 as transforms
import pandas as pd
import numpy as np
import torch.nn.functional as F
import random
import matplotlib.pyplot as plt

In [2]:
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

set_seed(SEED)

# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # wymagane przez niektóre operacje CUDA
device = torch.device("cuda")

In [3]:
def calc_accuracy(predictions: np.ndarray, targets: np.ndarray, n_classes=50):
    assert len(predictions) == len(targets)
    accuracies = []
    for i in range(n_classes):
        accuracies.append((predictions[targets == i] == i).sum() / (targets == i).sum())
    return np.mean(accuracies)

In [4]:
base_transform = transforms.Compose(
    [
        transforms.ToImage(),
        transforms.ToDtype(torch.float32, scale=True),
    ])

In [5]:
full_trainset = torchvision.datasets.ImageFolder("train/", transform=base_transform)

train_ratio = 0.7
full_trainset_size = len(full_trainset)
train_size = int(train_ratio * full_trainset_size)
validation_size = full_trainset_size - train_size
trainset, valset = torch.utils.data.random_split(
    full_trainset, [train_size, validation_size]
)

In [6]:
temp_transform = transforms.Compose(
    [
        transforms.Resize(64),
    ]
)

In [7]:
# all_samples = torch.stack([temp_transform(sample[0]) for sample in trainset])
# mean = all_samples.mean(axis=(0,2,3))
# std=all_samples.std(axis=(0,2,3))
# print(f"Data norm per channel: mean={mean} | std={std}")
# all_samples = None

In [8]:
mean = torch.tensor([0.5214, 0.4961, 0.4388])
std = torch.tensor([0.2841, 0.2772, 0.2978])

In [9]:
class WrappedSubset:
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        self.classes = subset.dataset.classes
        self.targets = [subset.dataset.targets[i] for i in subset.indices]

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

In [10]:
train_transform = transforms.Compose(
    [
        transforms.Resize(72),
        transforms.RandomCrop(64),
        transforms.RandomHorizontalFlip(),
        transforms.Normalize(mean=mean, std=std),
    ]
).to(device)

test_transform = transforms.Compose(
    [
        transforms.Resize(64),
        transforms.CenterCrop(64),
        transforms.Normalize(mean=mean, std=std),
    ]
).to(device)

trainset = WrappedSubset(trainset
# ,transform=train_transform
)
valset = WrappedSubset(valset
# , transform=test_transform
)


In [11]:
batch_size = 32
shuffle = True
num_workers = 12
prefetch_factor = 8
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=shuffle,
    num_workers=num_workers,
    pin_memory=True,
    prefetch_factor=prefetch_factor,
)
valloader = torch.utils.data.DataLoader(
    valset,
    batch_size=batch_size,
    shuffle=shuffle,
    num_workers=num_workers,
    pin_memory=True,
    prefetch_factor=prefetch_factor,
)

In [14]:
class ImprovedNet(nn.Module):
    def __init__(self, num_classes=50):
        super().__init__()
        # Block 1: 3 channels -> 32 channels. Pool down to 32x32
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)

        # Block 2: 32 channels -> 64 channels. Pool down to 16x16
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Block 3: 64 channels -> 128 channels. Pool down to 8x8
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)

        # Classifier: 128 channels * 8 * 8 = 8192 features
        self.fc1 = nn.Linear(128 * 8 * 8, 256)
        self.dropout = nn.Dropout(0.5) # Helps prevent overfitting!
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [15]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x) # The "Skip" connection
        return F.relu(out)

class FinalNet(nn.Module):
    def __init__(self, num_classes=50):
        super().__init__()
        self.in_channels = 64
        self.prep = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        
        # Three Stages of abstraction
        self.layer1 = ResidualBlock(64, 64, stride=1)
        self.layer2 = ResidualBlock(64, 128, stride=2) # Downsample to 32x32
        self.layer3 = ResidualBlock(128, 256, stride=2) # Downsample to 16x16
        self.layer4 = ResidualBlock(256, 512, stride=2) # Downsample to 8x8
        
        self.dropout = nn.Dropout(0.3) # 0.3 is usually enough for GAP models
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.prep(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x) # Apply just before the final decision
        return self.fc(x)

In [16]:
import torch.optim as optim

set_seed(42)

criterion = nn.CrossEntropyLoss()
# net = Net().to(device)
# net = long_net
# net = ImprovedNet(num_classes=len(trainset.classes)).to(device)
net = FinalNet(num_classes=len(trainset.classes)).to(device)
# optimizer = optim.Adam(net.parameters(), lr=0.001)

net

FinalNet(
  (prep): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (layer1): ResidualBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (shortcut): Sequential()
  )
  (layer2): ResidualBlock(
    (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(128, eps=1e-05, moment

In [17]:
optimizer = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)

# Inside the loop, after optimizer.step():
# scheduler.step()

In [18]:

num_of_epochs = 20
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.01, 
                                                steps_per_epoch=len(trainloader), 
                                                epochs=num_of_epochs)

In [19]:
net.train()
for epoch in range(num_of_epochs):  # loop over the dataset multiple times

    running_loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        # inputs, labels = inputs.to(device), labels.to(device)
        inputs = train_transform(inputs)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        # print statistics
        running_loss += loss.item()

    print("[%d/%d] loss: %.3f" % (epoch + 1, num_of_epochs, running_loss / 2000))
    running_loss = 0.0

print("Finished Training")

[1/20] loss: 2.946
[2/20] loss: 2.520
[3/20] loss: 2.256
[4/20] loss: 2.043
[5/20] loss: 1.873
[6/20] loss: 1.741
[7/20] loss: 1.621
[8/20] loss: 1.520
[9/20] loss: 1.429
[10/20] loss: 1.345
[11/20] loss: 1.262
[12/20] loss: 1.176
[13/20] loss: 1.085
[14/20] loss: 0.989
[15/20] loss: 0.879
[16/20] loss: 0.777
[17/20] loss: 0.678
[18/20] loss: 0.597
[19/20] loss: 0.548
[20/20] loss: 0.522
Finished Training


In [20]:
correct = 0
total = 0
net.eval()
with torch.no_grad():
    for inputs, labels in valloader:
        inputs = inputs.to(device, non_blocking=True)
        # inputs, labels = inputs.to(device), labels.to(device)
        inputs = test_transform(inputs)
        outputs = net(inputs).cpu()

        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(
    "Accuracy : %d %%" % (100 * correct / total)
)

Accuracy : 63 %


In [21]:
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in trainloader:
        inputs = inputs.to(device, non_blocking=True)
        # inputs, labels = inputs.to(device), labels.to(device)
        inputs = train_transform(inputs)
        outputs = net(inputs).cpu()

        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(
    "Accuracy : %d %%" % (100 * correct / total)
)

Accuracy : 88 %


In [25]:
# Define the path where you want to save
PATH = "fin_net_v2.pth"

# It is best practice to move the model to CPU before saving 
# to ensure it can be loaded on any device later.
torch.save(net.state_dict(), PATH)
print(f"Model weights saved to {PATH}")

Model weights saved to fin_net_v2.pth
